# Notebook 08 - Capstone: extract EVERY digit in an image

*Time: about 60 minutes*

## What you will learn
- Put the whole course together on a **realistic, photo-like picture**: uneven light, noise, numbers of different sizes at different positions.
- **Group boxes into rows** (by `y`), then sort each row left to right (by `x`).
- Read every digit, rebuild every number and **draw the result on the picture**.

## Prerequisites
- Notebooks 03-06 (adaptive threshold, morphology, contours, template matching, `int("".join(...))`).

## The plan
`gray -> blur -> adaptive threshold -> close -> contours -> area filter -> rows -> sort -> recognise -> numbers`

### What this code does

We load the libraries.

In [ ]:
import sys                                    # lets us change where Python looks for files
sys.path.append("..")                         # also look one folder up (utils.py lives there)
import cv2                                    # OpenCV
import numpy as np                            # NumPy: kernels and canvases
from utils import load, show, show_many       # our helpers
print("Libraries loaded - ready to go!")      # a visible sign that everything worked

### What you should see / Why we care

`Libraries loaded - ready to go!`

### What this code does

`IMAGE_PATH` is the picture to read (**change this line to use your own image**). We load it and look at it. Notice the dark corners (vignette) and numbers of different sizes.

In [ ]:
IMAGE_PATH = "../assets/digits_grid_photo_like.png"   # <- change me
img = load(IMAGE_PATH)                                # colour picture
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)          # gray version for processing
print("picture shape:", img.shape)                    # height, width, channels
show(img, "the photo-like picture", size=(9, 6))

### What you should see / Why we care

`picture shape: (600, 900, 3)`. A gray, slightly dirty-looking picture with five dark numbers scattered around.

## Step 1: make the digits white on black

### What this code does

We blur slightly, then use an **adaptive threshold** (Notebook 03: it copes with the dark corners). `THRESH_BINARY_INV` flips the result in one go, so the digits come out **white (255)** on black.

In [ ]:
blur = cv2.GaussianBlur(gray, (5, 5), 0)                   # remove a little sensor noise
thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY_INV, 51, 10)   # local decision, inverted
print("values in thresh:", np.unique(thresh))             # only 0 and 255
show_many([gray, thresh], ["gray", "adaptive threshold (white digits)"], columns=1, size=(9, 11))

### What you should see / Why we care

`values in thresh: [  0 255]` and clean white digits on black, even in the dark corners.

### What this code does

**Morphological close** (Notebook 03) joins any tiny breaks inside the strokes, so each digit stays in one piece.

In [ ]:
kernel = np.ones((3, 3), np.uint8)                         # a 3x3 grid of ones
closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel) # dilate then erode
print("white pixels before:", cv2.countNonZero(thresh), "| after:", cv2.countNonZero(closed))
show(closed, "after closing", size=(9, 6))

### What you should see / Why we care

The two counts differ by only a little. The picture looks the same: the digits were already solid, so closing is cheap insurance.

## Step 2: find the digits

### What this code does

Contours, then an **area filter** (`MIN_AREA = 80`) to drop any leftover specks, then store each box as `(x, y, w, h)`.

In [ ]:
MIN_AREA = 80                                              # smaller than this is not a digit
contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
boxes = []                                                 # list of (x, y, w, h)
for contour in contours:
    if cv2.contourArea(contour) >= MIN_AREA:               # big enough?
        boxes.append(cv2.boundingRect(contour))
print("contours:", len(contours), "| digit boxes kept:", len(boxes))

### What you should see / Why we care

`contours: 15 | digit boxes kept: 15`. There are 2 + 4 + 3 + 4 + 2 = **15 digits** in the picture.

## Step 3: sort into rows

Sorting by `x` alone would mix the lines together. Instead:
1. Compute each box's **vertical centre** `y + h / 2`.
2. Sort boxes by that centre (top to bottom).
3. Walk through them: when the centre jumps by more than `ROW_GAP` pixels, a **new row** begins.
4. Sort each row by `x` (left to right).

### What this code does

We build a list of `(centre_y, x, y, w, h)` tuples. Because the centre comes **first**, `sorted()` orders them top to bottom.

In [ ]:
centres = []                                               # (centre_y, x, y, w, h)
for x, y, w, h in boxes:
    centre_y = y + h / 2                                   # the vertical middle of the box
    centres.append((centre_y, x, y, w, h))
centres = sorted(centres)                                  # top to bottom
print("centre_y of every box, top to bottom:")
for item in centres:
    print(round(item[0]), end=" ")                         # end=" " keeps them on one line
print()

### What you should see / Why we care

A list of 15 numbers that jumps in **three clusters** (around 100, 300 and 480), one cluster per row.

### What this code does

We walk through the sorted list and start a new row when the centre jumps by more than `ROW_GAP = 40` pixels. Then each row is sorted left to right.

In [ ]:
ROW_GAP = 40                                               # a jump bigger than this = new row
rows = []                                                  # finished rows
current = []                                               # the row we are building
last_y = None                                              # centre of the previous box
for centre_y, x, y, w, h in centres:
    if last_y is not None and centre_y - last_y > ROW_GAP: # big jump: new row
        rows.append(sorted(current))                       # finish the row, left to right
        current = []
    current.append((x, y, w, h))                           # add this box to the row
    last_y = centre_y
rows.append(sorted(current))                               # the last row
for number in range(len(rows)):
    print("row", number + 1, "has", len(rows[number]), "digits, first x =", rows[number][0][0])

### What you should see / Why we care

`row 1 has 6 digits`, `row 2 has 3 digits`, `row 3 has 6 digits`: the digits are grouped by line and sorted left to right inside each line.

## Step 4: recognise every digit

We reuse the template matcher from Notebook 06 (same code).

### What this code does

`tidy` squashes a crop to 32 x 32 and `templates` holds the ten digit pictures drawn with `cv2.putText`.

In [ ]:
FONT = cv2.FONT_HERSHEY_SIMPLEX                            # font for templates
DIGITS = "0123456789"

def tidy(crop):                                            # squash to 32x32
    return cv2.resize(crop, (32, 32), interpolation=cv2.INTER_AREA)

templates = {}                                             # digit -> picture
for digit in DIGITS:
    canvas = np.zeros((120, 120), dtype=np.uint8)          # black canvas
    cv2.putText(canvas, digit, (20, 90), FONT, 2.5, 255, 4, cv2.LINE_AA)
    _, canvas = cv2.threshold(canvas, 127, 255, cv2.THRESH_BINARY)
    x, y, w, h = cv2.boundingRect(canvas)                  # tight box
    templates[digit] = tidy(canvas[y:y + h, x:x + w])
print("templates ready:", len(templates))

### What you should see / Why we care

`templates ready: 10`

### What this code does

The matcher from Notebook 06: compare a crop with all ten templates and return the best digit, its score, and the runner-up.

In [ ]:
def recognize(crop, templates):                            # the matcher from Notebook 06
    small = tidy(crop)
    best, best_score, second, second_score = "?", -2.0, "?", -2.0
    for digit in templates:
        score = cv2.matchTemplate(small, templates[digit], cv2.TM_CCOEFF_NORMED)[0][0]
        if score > best_score:
            second, second_score = best, best_score
            best, best_score = digit, score
        elif score > second_score:
            second, second_score = digit, score
    return best, float(best_score), second, float(second_score)
test = recognize(closed[77:131, 63:105], templates)      # the first digit of row 1
print("recognize() ready, first digit looks like:", test[0], "confidence", round(test[1], 2))

### What you should see / Why we care

`recognize() ready, first digit looks like: 5 confidence 0.96`

### What this code does

We read every box, row by row, and save one record per digit: `(box id, row, x, y, w, h, digit, confidence)`. Digits are read from `closed` (the clean white-on-black picture). Then we print the digits **in reading order** with their positions.

In [ ]:
detections = []                                            # one record per digit
box_id = 0                                                 # running number
for row_number in range(len(rows)):
    for x, y, w, h in rows[row_number]:                    # left to right in this row
        digit, score, second, second_score = recognize(closed[y:y + h, x:x + w], templates)
        detections.append((box_id, row_number + 1, x, y, w, h, digit, score))
        box_id = box_id + 1
print("digits found, in reading order:")
for d in detections:
    print(d[6], end=" ")                                   # the digit itself
print("\ntotal:", len(detections))

### What you should see / Why we care

**`5 2 7 3 1 9 9 0 8 2 0 2 5 4 6`** (rows 1, 2 and 3, each read left to right), and `total: 15`.

## Step 5: rebuild the numbers

Inside a row, two digits belong to the **same number** when they are close together. A gap bigger than 80% of the digit height starts a **new number** (the idea from Notebook 06).

### What this code does

For each row we collect digits, split them at big gaps, and turn each group into an integer with `int("".join(...))`.

In [ ]:
all_numbers = []                                           # every number in the picture
for row_number in range(len(rows)):
    current, last_right, row_numbers = [], None, []        # digits so far / right edge / numbers
    for d in detections:
        if d[1] == row_number + 1:                         # only digits of this row
            x, w, h, digit = d[2], d[4], d[5], d[6]
            if last_right is not None and x - last_right > 0.8 * h:   # big gap: new number
                row_numbers.append(int("".join(current)))
                current = []
            current.append(digit)
            last_right = x + w
    row_numbers.append(int("".join(current)))              # the last number of the row
    print("Row", row_number + 1, ":", row_numbers)
    all_numbers = all_numbers + row_numbers
print("All numbers:", all_numbers)

### What you should see / Why we care

`Row 1 : [52, 7319]`, `Row 2 : [908]`, `Row 3 : [2025, 46]` and `All numbers: [52, 7319, 908, 2025, 46]`.

### What this code does

A built-in check: we compare with the numbers we drew (we know them, because we made the image ourselves).

In [ ]:
expected = [52, 7319, 908, 2025, 46]                       # what make_assets.py drew
print("expected:", expected)
print("found   :", all_numbers)
print("ALL CORRECT:", all_numbers == expected)             # True only if identical
print("sum of all numbers:", sum(all_numbers))

### What you should see / Why we care

`ALL CORRECT: True` and `sum of all numbers: 10350`.

## The hero picture

### What this code does

We draw on a **copy** of the original colour picture: a green rectangle around every detected digit and the predicted digit **written above it** with `cv2.putText`.

In [ ]:
result = img.copy()                                        # keep the original untouched
for box_id, row, x, y, w, h, digit, score in detections:
    cv2.rectangle(result, (x - 3, y - 3), (x + w + 3, y + h + 3), (0, 200, 0), 2)   # green box
    cv2.putText(result, digit, (x + w // 4, y - 10), FONT, 0.9, (0, 140, 0), 2, cv2.LINE_AA)
show(result, "HERO: every digit found, with its prediction above it", size=(12, 8))
print("drawn", len(detections), "rectangles and labels")

### What you should see / Why we care

The photo-like picture with **15 green rectangles**, each with the predicted digit above it. This is the whole course in one picture: pixels, thresholds, contours, templates, and numbers.

### What this code does

A summary table: box id, position and size, prediction and confidence.

In [ ]:
print("  id    x    y    w    h  digit   conf")             # column titles
for box_id, row, x, y, w, h, digit, score in detections:
    print("{:>4} {:>4} {:>4} {:>4} {:>4} {:>6} {:>6.2f}".format(box_id, x, y, w, h, digit, score))
lowest = 1.0                                                # start high
for d in detections:
    if d[7] < lowest:                                       # a weaker one?
        lowest = d[7]
print("lowest confidence:", round(lowest, 2))

### What you should see / Why we care

15 rows, with confidence values mostly above 0.9, and the final line `lowest confidence: 0.91`. All predictions are correct.

## WHERE THIS WOULD FAIL

This pipeline is tuned to this kind of picture. Expect trouble when:
- **Digits touch** each other: they form one contour and are read as one big (wrong) shape.
- **Digits are broken** into pieces (the area filter may delete the pieces, or each piece becomes a "digit").
- **Different fonts or handwriting** (Notebook 06): the templates no longer look like the digits.
- **Tilted or curved text**: boxes become wrong and the templates mismatch.
- **Letters or symbols** also appear: the reader will force them into the *nearest digit*. It has no "this is not a digit" answer. (Watch the low confidence.)
- **Strong shadows, glare, or very different lighting** require different `blockSize` and `C` values for the adaptive threshold.
- **Digits are much smaller** than `MIN_AREA`, or specks are bigger than it.

The honest summary: *template matching is a great teaching tool and works on controlled pictures. For the messy real world, use a trained OCR model* (Notebook 09).

## Recap
- **Rows:** group by `y` centre, then sort each row by `x`.
- **Numbers:** split at big gaps and use `int("".join(digits))`.
- **Check:** compare with what you expect, and watch the confidence.
- To read your own picture, change `IMAGE_PATH` at the top and run again. Remember the limits above.

**Next (optional):** Notebook 09 compares our reader with a real OCR engine.